# Installing packages and loading data file

In [2]:
# Install packages 
#%pip install pandas numpy matplotlib scikit-learn ydata-profiling

In [3]:
# Import relavent packages 
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
import matplotlib.pyplot as plt

In [4]:
# Load cleaned dataset
df_clean = pd.read_csv("/Users/muyuanli/Library/CloudStorage/OneDrive-CornellUniversity/Desktop/ANSC 6040/df_clean.csv")


# Splitting dataset for Random Forest
## Stratify by cow ID and split by time 

In [5]:
# Checking how many records on average a cow has
df_clean.groupby("AnimalId")["EventDate"].agg(["min", "max", "count"]).describe()

,count
count,9011.000000
mean,669.522916
std,448.709327
min,1.000000
25%,276.000000
50%,631.000000
75%,1045.500000
max,1673.000000


In [6]:
# Checking data type
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6033071 entries, 0 to 6033070
Data columns (total 14 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   AnimalId                float64
 1   LactationNumber         float64
 2   DaysInMilk              float64
 3   ReproductionStatus      object 
 4   EventDate               object 
 5   AvgMilkFlow             float64
 6   Flow30_60Session        float64
 7   YieldFirst2Min_Session  float64
 8   YieldSession            float64
 9   DurationSession_Sec     int64  
 10  Milking                 int64  
 11  z_AvgMilkFlow           float64
 12  z_Flow30_60Session      float64
 13  z_YieldSession          float64
dtypes: float64(10), int64(2), object(2)
memory usage: 644.4+ MB


In [7]:
# Convert variables to correct forms
df_clean["EventDate"] = pd.to_datetime(df_clean["EventDate"])
df_clean["AnimalId"] = df_clean["AnimalId"].astype(int)

# Consider making a smaller dataset for model troubleshooting

In [8]:
# Sort dataset by cow ID then EventDate
df_sort = df_clean.sort_values(["AnimalId", "EventDate"])

In [9]:
# Defining train/stest status for each cow record
def split_by_time(group, test_frac=0.2):
    n_test = max(1, int(len(group) * test_frac))
    return pd.Series(["train"] * (len(group) - n_test) + ["test"] * n_test,
                      index=group.index)

In [10]:
# Creating new column called "split" to decide which record goes into train or test
df_sort["split"] = df_sort.groupby("AnimalId", group_keys=False).apply(split_by_time)

/var/folders/5n/f_9m5grn7vq6mx7m6n8bwb6c0000gn/T/ipykernel_18354/2498999006.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sort["split"] = df_sort.groupby("AnimalId", group_keys=False).apply(split_by_time)


In [11]:
# Creating training and testing sets
df_train = df_sort[df_sort["split"] == "train"]
df_test = df_sort[df_sort["split"] == "test"]

In [12]:
# Checking
print(df_train.shape, df_test.shape)
print(df_train["AnimalId"].nunique(), df_test["AnimalId"].nunique())

(4830001, 15) (1203070, 15)
8985 9011


In [13]:
# Creating features and labels
features = ["AvgMilkFlow", "Flow30_60Session", "YieldFirst2Min_Session",
            "YieldSession", "DurationSession_Sec", "Milking"]

X_train = df_train[features]
y_train = df_train["AnimalId"]

X_test = df_test[features]
y_test = df_test["AnimalId"]

In [14]:
# Default model

rf_default = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,          # caps how deep trees can grow
    min_samples_leaf=2,    # prevents tiny, memory-hungry leaves
    n_jobs=-1,
    random_state=0
)
rf_default.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_leaf=2, n_estimators=50,
                       n_jobs=-1, random_state=0)

In [15]:
# Kernel kept crashing, trying the model on a smaller test set 
small_test = X_test.sample(5000, random_state=0)
y_pred_small = rf_default.predict(small_test)
print("worked fine on a small sample")

worked fine on a small sample


In [16]:
batch_size = 10000
predictions = []

for start in range(0, len(X_test), batch_size):
    end = start + batch_size
    batch_pred = rf_default.predict(X_test.iloc[start:end])
    predictions.append(batch_pred)

y_pred = np.concatenate(predictions)

In [17]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.0036805838396768266
